# Yorùbá OCR Research — Google Colab Pipeline

**Runtime:** GPU (T4 or better) → Runtime → Change runtime type → T4 GPU

> **Data:** Keep `data/` on Drive under the repo folder (`My Drive/yoruba_ocr_research/data/`). Code tracks GitHub; work in the Drive repo root so paths match `data/processed/`.

**Pipeline (shell phases):**
1. Mount Drive & repo
2. Install deps (Paddle GPU + `requirements.txt`; for **PaddleOCR-VL-1.5** also `pip install 'transformers>=5' peft` for phases 15–16)
3. Clone **PaddleOCR** (PP-OCRv4 **comparison** train/eval; same crops feed VL export)
4. Phase **01** consolidate (if needed) → **02–03** analyse + config
5. Phase **04** PP-OCRv4 CRNN fine-tune (**classical comparison**)
6. Phases **05–08** Paddle eval, Tesseract, Qwen, **PP-OCRv4 ablations** (long)
7. **Primary supervised model (Table 1):** **14** export JSONL → **15** zero-shot VL (`SKIP_VL15_EVAL=0`) → **16** LoRA → **15** with `--adapter-path` for main metrics — see `docs/vl15_pipeline.md`
8. **Optional:** phases **12** / **13** diagnostics & strict alignment
9. Phase **09** compile tables + `eval_alignment_report.json`
10. Phase **99** backup to Drive

**Not in repo:** Mistral OCR. HF model id: `PaddlePaddle/PaddleOCR-VL-1.5`.

### How to run this notebook

- **Google Colab:** Runtime → Change runtime type → **GPU (T4+)**. Run cells **top to bottom** from Step 1. Data must live under `My Drive/yoruba_ocr_research/` as in Step 1.
- **Local (Jupyter / VS Code / Cursor):** Open `yor_ocr.ipynb` from the **repo root** (`yoruba_ocr_research/`). Activate your venv (`source .venv/bin/activate`), install `requirements.txt`, ensure `data/processed/` exists. **Skip** the Drive mount and `git clone` cells; `cd` to the repo in a terminal cell if needed (`os.chdir('/path/to/yoruba_ocr_research')`). On **macOS**, skip the `paddlepaddle-gpu` pip line and use CPU Paddle or install per Paddle docs; PP-OCR training will be slower.
- **Primary supervised fine-tune** = **PaddleOCR-VL-1.5 LoRA** (phases **14 → 15 → 16 → 15** with adapter). Phase **04** is PP-OCRv4 **comparison** only. Generated PP-OCR config uses **`epoch_num: 40`** by default (was 100); override with `CONFIG_EPOCHS` / `TRAIN_EPOCHS_OVERRIDE` if needed.\n\n### Integrity & metrics conventions (post P1–P6)\n\n- Step 4 now runs **`scripts/02b_data_quality_audit.py`** (read-only) after `verify-data` to confirm the Phase-01 hygiene filters produced clean single-line crops (`data_quality.json`).\n- Step 8 runs **`scripts/12_diagnose_hypotheses.py checkpoints`** after `phase_09_compile.sh` to verify every `metrics.csv` row has a matching `meta.json` sidecar and no Paddle checkpoint silently re-initialised its CTC head.\n- `metrics.csv` / `metrics_summary.csv` carry four-valued **`phantom`** (`true`/`false`/`n/a`/`unknown`) and corpus-level **`der`** alongside **`der_n`** (samples with ≥1 GT diacritic) and **`der_insertion_rate`** (precision-side signal for diacritic hallucination). See `docs/metrics_conventions.md`.\n- Pre-rerun artifacts live under `results/tables/archive/pre_rerun_2026-04-21/`; pre-P1 `metrics.csv` under `archive/pre_integrity/`; pre-P6 `metrics.csv` under `archive/pre_der_conditional/`.

## Step 1 — Mount Drive & Set Up Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
REPO_DIR   = f'{DRIVE_ROOT}/yor_ocr_research'
DATA_DIR   = f'{REPO_DIR}/data'


print(f'Drive mounted.')
print(f'Repo + data directory : {REPO_DIR}')
print(f'data/processed exists : {os.path.isdir(DATA_DIR + "/processed")}')
print(f'data/raw exists       : {os.path.isdir(DATA_DIR + "/raw")}')

In [ ]:
import os, shutil

# Ensure we are in the right base directory
os.chdir('/content/drive/MyDrive')

# If scripts/ is missing, the previous clone is broken and needs to be wiped
if os.path.isdir(REPO_DIR) and not os.path.exists(f'{REPO_DIR}/scripts'):
    print(f'Detected broken repo at {REPO_DIR}. Wiping for a clean clone...')
    shutil.rmtree(REPO_DIR)

if not os.path.isdir(REPO_DIR):
    print(f'Cloning repo to {REPO_DIR}...')
    !git clone --depth 1 https://github.com/sam4rano/yoruba_ocr_research.git "{REPO_DIR}"
else:
    print('Repo exists and appears valid — updating...')
    !git -C "{REPO_DIR}" fetch --depth 1 origin main
    !git -C "{REPO_DIR}" reset --hard origin/main

# Set working directory to repo root
os.chdir(REPO_DIR)
!pwd
!ls -F

## Step 2 — Install Python Dependencies

In [ ]:
# Install PaddlePaddle GPU (Colab Linux x86 with CUDA)
!python -m pip install paddlepaddle-gpu -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q

In [ ]:
# Install all other project requirements (skip paddlepaddle line — already installed above)
!grep -v '^paddlepaddle' requirements.txt | grep -v '^#' | grep -v '^$' > /tmp/reqs_no_paddle.txt
!python -m pip install -r /tmp/reqs_no_paddle.txt -q
print('All dependencies installed.')

In [ ]:
# Verify paddle is importable and sees the GPU
import paddle
print('PaddlePaddle version:', paddle.__version__)
print('GPU available:', paddle.device.is_compiled_with_cuda())
print('Device count :', paddle.device.cuda.device_count())

## Step 3 — Clone PaddleOCR Repo

In [ ]:
import os, shutil

# If the directory exists but is missing requirements, it's likely a failed clone
if os.path.isdir('PaddleOCR') and not os.path.exists('PaddleOCR/requirements.txt'):
    print('Removing corrupted PaddleOCR directory...')
    shutil.rmtree('PaddleOCR')

if not os.path.isdir('PaddleOCR'):
    # Explicitly clone the main branch to avoid 'nonexistent ref' errors
    print('Cloning PaddleOCR main branch...')
    !git clone --depth 1 -b main https://github.com/PaddlePaddle/PaddleOCR.git PaddleOCR
else:
    print('PaddleOCR already cloned and verified — skipping.')

# PaddleOCR repo requirements (detection/recognition training)
if os.path.exists('PaddleOCR/requirements.txt'):
    !python -m pip install -q -r PaddleOCR/requirements.txt
else:
    print('Error: PaddleOCR/requirements.txt still not found. Check if the clone was successful.')

# Qwen 2.5 VL baselines
!python -m pip install -q accelerate qwen-vl-utils bitsandbytes
# PaddleOCR-VL-1.5 (HF) — phases 15–16; use transformers v5 per model card
!python -m pip install -q 'transformers>=5.0.0' peft
print('PaddleOCR repo + VL/Qwen extras installed.')

try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        os.environ['HF_TOKEN'] = _tok
        os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', _tok)
        print('HF_TOKEN loaded from Colab secret (transformers will authenticate).')
    else:
        print('No HF_TOKEN Colab secret set — public models will still download anonymously.')
except Exception:
    if os.environ.get('HF_TOKEN'):
        print('HF_TOKEN already present in environment.')
    else:
        print('Not running in Colab; HF_TOKEN not set.')

## Step 4 — Rebuild `data/processed/`, Verify, Analyse & Config (Phases 01–03)

The next cell runs **Phase 01** (`scripts/01_consolidate_data.py`) to rebuild `data/processed/` from every `data/raw/yoruba_ocr_NNNNN*/` export, applying all hygiene filters:

- single source of truth for the Yorùbá Unicode whitelist: `scripts/yoruba_charset.py`
- label length bounds `[3, 100]` chars, image height `≤ 180 px`, char dict rebuilt from cleaned labels
- English-noise rejection (stopwords, months, roman numerals, `c q v x z`, `...`, `___`, `=`)
- per-filter drop counts are written to `results/tables/consolidation_report.json` for the paper

**Whenever `data/raw/` has changed** (new exports, removed/renamed samples, relabelled crops), set `RESET_PROCESSED=1` before running so orphan PNGs from deleted raw samples are pruned:

```python
%env RESET_PROCESSED=1
```

(The consolidate script only overwrites; it never deletes. Labels and the char dict are always rewritten cleanly, so the wipe is only needed to prune orphan image files.)

The cell then runs `verify-data` checks, **`scripts/02b_data_quality_audit.py`** (read-only hygiene re-check), Phase 02 analysis, and Phase 03 config generation.

In [ ]:
import os, subprocess, shutil, json
from pathlib import Path

# Ensure data/raw exists so the consolidation script doesn't crash
os.makedirs('data/raw', exist_ok=True)

# Surface the audit trail the paper needs
rep = Path('results/tables/consolidation_report.json')
if rep.is_file():
    r = json.loads(rep.read_text(encoding='utf-8'))
    sc = r.get('split_counts', {})
    h = r.get('hygiene', {})
    print(
        f"\n[consolidation] merged {r.get('n_exports')} export(s) "
        f"→ {r.get('unique_images_total')} unique crops "
        f"(train={sc.get('train', 0)}, val={sc.get('val', 0)}, test={sc.get('test', 0)}) "
        f"char_dict_size={r.get('char_dict_size')}"
    )
else:
    print('[consolidation] WARNING: consolidation_report.json not found — Phase 01 may have failed if data/raw was empty.')

# Verify the rebuilt folders are complete
checks = {
    'data/processed/labels/train.txt': 'file',
    'data/processed/labels/val.txt':   'file',
    'data/processed/labels/test.txt':  'file',
    'data/processed/dictionary/yoruba_char_dict.txt': 'file',
    'data/processed/images/train':     'dir',
    'data/processed/images/val':       'dir',
    'data/processed/images/test':      'dir',
}

ok = True
for path, kind in checks.items():
    exists = os.path.isfile(path) if kind == 'file' else os.path.isdir(path)
    if exists:
        if kind == 'dir':
            n = len(os.listdir(path))
            print(f'  ✅  {path}  ({n} items)')
        else:
            n = int(subprocess.check_output(['wc', '-l', path]).split()[0])
            print(f'  ✅  {path}  ({n} lines)')
    else:
        print(f'  ❌  MISSING: {path}')
        ok = False

if not ok:
    print('\n[!] data/processed is incomplete. If this is a fresh clone, you need to upload your raw export folders into data/raw/ first.')
else:
    print('\n✅  All data checks passed — ready to train!')

In [ ]:
# P2 data-quality audit (read-only): profiles labels/images under data/processed and

!python scripts/02b_data_quality_audit.py \
    --data-dir data/processed \
    --out-json results/tables/data_quality.json

import json
q = json.load(open('results/tables/data_quality.json'))
print(f"thresholds : {q.get('thresholds')}")
for split, prof in q.get('per_split', {}).items():
    if not prof.get('n'):
        continue
    wd = prof['would_drop']
    print(
        f"{split:5s}  n={prof['n']:5d}  "
        f"short={wd['label_too_short']['count']:4d}  "
        f"long={wd['label_too_long']['count']:4d}  "
        f"tall={wd['image_too_tall']['count']:4d}  "
        f"non-yor={wd['non_yoruba_codepoint']['count']:4d}"
    )
tot = q.get('totals', {}).get('would_drop', {})
if tot:
    print(f"\n[would-drop totals] {tot}")
    print("(Non-zero values mean the hygiene filters in 01_consolidate_data.py are needed; re-run Phase 01 with the flags if anything is off.)")

In [ ]:
# Phase 02 — Dataset analysis (Generates the plots for the paper)
!python3 scripts/02_analyze_dataset.py --data-dir data/processed --output-dir results/tables --plot

# Phase 03 — PP-OCR YAML
!CONFIG_FORCE_GPU=1 bash scripts/shell/phase_03_config.sh

print('Phases 02 (with plots) and 03 complete.')

In [ ]:
# Sanity check: show key fields from the generated config
import yaml
with open('configs/paddleocr_yoruba_rec.yml') as f:
    cfg = yaml.safe_load(f)

g = cfg.get('Global', {})
print('use_gpu  :', g.get('use_gpu'))
print('char_dict:', g.get('character_dict_path'))
print('epochs   :', g.get('epoch_num'))
print('save_dir :', g.get('save_model_dir'))

## Step 5 — *(Optional)* Fine-Tune PP-OCRv4 CRNN on GPU (Phase 04)

**Skip this step if your paper only fine-tunes PaddleOCR-VL-1.5.** The primary supervised row in Table 1 comes from Phase 16 (VL-1.5 LoRA), not from PP-OCRv4.

Run this cell **only** if you want the classical CRNN comparison (adds a `finetuned_paddleocr_v1` row alongside `baseline_english_pretrained`). Otherwise leave `SKIP_PADDLE_TRAIN=1` below and move on to Step 6 — Phase 05 will still produce the zero-shot `baseline_english_pretrained` row for the "PaddleOCR" baseline in your abstract.

In [ ]:
import os

# ---------------------------------------------------------------------------
# Phase 04 — PP-OCRv4 CRNN fine-tune (OPTIONAL)
# ---------------------------------------------------------------------------
# Default: SKIP. Your paper fine-tunes only PaddleOCR-VL-1.5 (Phase 16).
# Set SKIP_PADDLE_TRAIN='0' below if you want the classical CRNN comparison row.
os.environ['SKIP_PADDLE_TRAIN'] = os.environ.get('SKIP_PADDLE_TRAIN', '0')

if os.environ['SKIP_PADDLE_TRAIN'] == '0':
    print('SKIP_PADDLE_TRAIN=1 — skipping PP-OCRv4 fine-tune. '
          'Phase 05 will still produce the zero-shot `baseline_english_pretrained` row.')
else:
    os.environ['EVAL_USE_GPU']     = '1'
    os.environ['CONFIG_FORCE_GPU'] = '1'
    os.environ['TRAIN_RESUME']     = '1'  # auto-resume from latest checkpoint in experiments/finetuned/
    # Optional overrides (PP-OCR CRNN only — not VL-1.5):
    os.environ['TRAIN_EPOCHS_OVERRIDE'] = '40'
    # os.environ['TRAIN_BATCH_OVERRIDE']  = '64'
    get_ipython().system('bash scripts/shell/phase_04_train.sh')

## Step 6 — Evaluate zero-shot baselines (Phases 05–07)

All rows produced here are **zero-shot** (no fine-tuning on Yorùbá):

- **Phase 05** — PaddleOCR: `baseline_english_pretrained` row (English PP-OCRv3 rec, English dict). Fine-tuned PP-OCRv4 row is skipped automatically unless you enabled Step 5.
- **Phase 06** — Tesseract (`tesseract-ocr-yor` language pack).
- **Phase 07** — Qwen 2.5-VL zero-shot (multimodal baseline).
- **Phase 08** — ablations; only meaningful if you ran Step 5. Off by default (`SKIP_ABLATION=1`).

The **primary supervised row** (`paddleocr_vl15_lora_finetuned`) is produced later in Step 7c.

In [ ]:
# Phase 05: PaddleOCR zero-shot baseline (English PP-OCRv3 rec, English dict)
# We set ALLOW_HEAD_REINIT=1 to explicitly allow metrics recording despite the head mismatch.
import os

!ALLOW_HEAD_REINIT=1 EVAL_USE_GPU=1 python3 scripts/05_evaluate.py \
    --model-dir experiments/baseline/pretrained/en_PP-OCRv3_rec_train \
    --data-dir data/processed \
    --split test \
    --model-name baseline_english_pretrained \
    --rec-config configs/paddleocr_yoruba_rec.yml \
    --use-gpu \
    --paddle-dir PaddleOCR \
    --allow-head-reinit

In [ ]:
# Install Tesseract system binary + Yoruba language pack
!apt-get install -qq tesseract-ocr tesseract-ocr-yor 2>/dev/null

# Phase 06: Tesseract baseline
!bash scripts/shell/phase_06_tesseract.sh

In [ ]:
# Phase 07: Qwen 2.5-VL zero-shot baseline
# Needs ~16GB VRAM. 4-bit quantize is enabled by default in the script.
import os
os.environ['SKIP_QWEN'] = '0'  # Explicitly enable Qwen (skipped by default otherwise)
!bash scripts/shell/phase_07_qwen.sh


In [ ]:
# Phase 08 — Ablation study (Now enabled to test data size impacts)
os.environ['SKIP_ABLATION'] = '0'
!bash scripts/shell/phase_08_ablation.sh

### Primary supervised — PaddleOCR-VL-1.5 (HF)

This is the **main fine-tuned system** for Yorùbá line crops (LoRA on **16**, eval on **15**). PP-OCRv4 phase **04** is the classical CRNN comparison, not the primary supervised row in Table 1.

- **14** exports JSONL under `data/paddleocr_vl15_sft/` (does not alter `data/processed/`).
- **15** zero-shot VL — set `SKIP_VL15_EVAL=0` (GPU); baseline before LoRA.
- **16** LoRA — long GPU job; default **`VL15_LORA_EPOCHS=1`** per epoch; optional **`VL15_GRAD_ACCUM=4`**. Then **15** with `--adapter-path experiments/paddleocr_vl15_lora/adapter` for `paddleocr_vl15_lora_finetuned`.

**Optional QA:** **12** hypothesis checks; **13** strict alignment: `VERIFY_STRICT=1 bash scripts/shell/phase_13_verify_eval.sh`.

See `docs/vl15_pipeline.md` and `docs/research_standards_review.md`.

In [ ]:
# ===============================================================
# Step 7a — Phase 14: Export VL-1.5 SFT JSONL from data/processed/

!bash scripts/shell/phase_14_export_vl15.sh

# Sanity: confirm the export succeeded before burning any GPU time.
from pathlib import Path
import json

export_dir = Path('data/paddleocr_vl15_sft')
for split in ('train', 'val', 'test'):
    f = export_dir / f'{split}.jsonl'
    n = sum(1 for _ in f.open(encoding='utf-8')) if f.is_file() else 0
    flag = '✅' if n > 0 else '❌'
    print(f'  {flag}  {split}.jsonl : {n} samples')

m = export_dir / 'manifest.json'
if m.is_file():
    mf = json.loads(m.read_text(encoding='utf-8'))
    print(
        f'\n  manifest : prompt_hash={mf.get("prompt_hash", "?")[:12]} '
        f'dict_path={mf.get("dict_path")} '
        f'created={mf.get("created_utc")}'
    )
else:
    print('\n  ❌ manifest.json missing — Phase 14 did not finish cleanly.')

In [ ]:
# ===============================================================
# Step 7b — Phase 15: VL-1.5 ZERO-SHOT eval (baseline row for Table 1)
# ===============================================================
import os
import pandas as pd

os.environ['SKIP_VL15_EVAL']     = '0'
os.environ['VL15_QUANTIZE_4BIT'] = '1'

!bash scripts/shell/phase_15_eval_vl15.sh

# Confirm a zero-shot row landed in metrics.csv
metrics_path = 'results/tables/metrics.csv'
if os.path.exists(metrics_path):
    df = pd.read_csv(metrics_path)
    print(f'Columns found in CSV: {df.columns.tolist()}')

    # Use 'model' or 'model_name' based on what is actually in the CSV
    col_name = 'model_name' if 'model_name' in df.columns else 'model'

    if col_name in df.columns:
        zs = df[df[col_name] == 'paddleocr_vl15_zero_shot']
        if zs.empty:
            print('\n❌ No paddleocr_vl15_zero_shot row yet — check the log above.')
        else:
            cols = [c for c in ('split', 'n', 'cer', 'wer', 'der', 'der_n', 'phantom') if c in zs.columns]
            print('\n✅ VL-1.5 zero-shot row(s):')
            print(zs[cols].tail().to_string(index=False))
    else:
        print(f'\n❌ Neither "model_name" nor "model" found in {metrics_path}')
else:
    print(f'\n❌ {metrics_path} does not exist.')

In [ ]:
# ===============================================================
# Step 7c — Phase 16: LoRA fine-tune VL-1.5, then re-eval with adapter
# ===============================================================
import os

# 1. Configure for full training
os.environ['VL15_LORA_EPOCHS'] = '5'
os.environ['VL15_GRAD_ACCUM']  = '4'
# Remove the 50-sample limit to use all ~2300 images
if 'VL15_LORA_MAX_SAMPLES' in os.environ:
    del os.environ['VL15_LORA_MAX_SAMPLES']

print("Starting full fine-tuning on entire dataset...")
!bash scripts/shell/phase_16_train_vl15_lora.sh

print("\nDone! Now you should re-upload the adapter to Hugging Face or run the inference cell again to see the improvement.")

In [ ]:
# Step 7d — Evaluate the FINE-TUNED model using the LoRA adapter
import os

os.environ['SKIP_VL15_EVAL'] = '0'

# Run eval with the --adapter-path flag to use your trained weights
# Removed --model-name as it is not a valid argument for this script
!python3 scripts/15_baseline_paddleocr_vl15.py \
    --data-dir data/processed \
    --split test \
    --results-csv results/tables/metrics.csv \
    --adapter-path experiments/paddleocr_vl15_lora/adapter \
    --quantize-4bit

# Re-run Phase 09 to compile the new row into the summary table
!bash scripts/shell/phase_09_compile.sh

In [ ]:
import pandas as pd
df_final = pd.read_csv('results/tables/metrics_summary.csv')
display(df_final[['display_name', 'cer_pct', 'wer_pct', 'der_pct']])

## Step 8 — Compile results (Phase 09)

In [ ]:
!bash scripts/shell/phase_09_compile.sh

# Table 1: metrics_summary.csv == table1_main_comparison.csv
# eval_alignment_report.json — non-empty "mismatches" => re-run evals on current data

import json
from pathlib import Path
import pandas as pd

df = pd.read_csv('results/tables/metrics_summary.csv')
print(df.to_string(index=False))

# Surface any phantom rows up-front so they cannot be mistaken for real metrics.
if 'phantom' in df.columns:
    n_phantom = int((df['phantom'] == 'true').sum())
    if n_phantom:
        print(f"\n[integrity] {n_phantom} row(s) flagged phantom=true — DO NOT report these numbers.")
        print(df[df['phantom'] == 'true'][['model_name', 'split', 'cer', 'wer', 'phantom']].to_string(index=False))
    else:
        print("\n[integrity] OK — no phantom rows.")

rep = Path('results/tables/eval_alignment_report.json')
if rep.is_file():
    r = json.loads(rep.read_text(encoding='utf-8'))
    if r.get('mismatches'):
        print('\n[align] Mismatch: stored n != current label counts — see', rep)
    else:
        print('\n[align] OK —', rep)

### Step 8b — Research Paper Artifacts (Visuals & Tables)

In [ ]:
import pandas as pd

# Display Table 1 for the paper
table1_path = 'results/tables/table1_main_comparison.csv'
if os.path.exists(table1_path):
    print('--- Table 1: Main Model Comparison ---')
    display(pd.read_csv(table1_path))
else:
    print('Table 1 not found. Run scripts/11_compile_results.py')

## Step 9 — Backup results to Google Drive (Phase 99)

In [ ]:
import os
# Results + model checkpoints go to a timestamped subfolder on your Drive.
os.environ['DRIVE_BACKUP_ROOT']    = f'{DRIVE_ROOT}/yoruba_ocr_backups'
os.environ['BACKUP_EXPERIMENTS']   = '1'  # also backup model checkpoints

!bash scripts/shell/phase_99_backup.sh
print('Backup complete. Check: My Drive/yoruba_ocr_backups/')

# Final Research Summary Report: Yorùbá OCR

This report summarizes the results of the fine-tuning pipeline for **PaddleOCR-VL-1.5** using LoRA on the Yorùbá dataset.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Load metrics
df_summary = pd.read_csv('results/tables/metrics_summary.csv')

# Format Table for Report
report_table = df_summary[['display_name', 'cer_pct', 'wer_pct', 'der_pct', 'n']].copy()
report_table.columns = ['Model Name', 'CER (%)', 'WER (%)', 'DER (%)', 'Sample Size (n)']

print("--- Experimental Findings: Model Comparison ---")
display(report_table)

# Highlight Best Model
best_row = df_summary.loc[df_summary['cer_pct'].idxmin()]

Markdown(f"""
### Key Findings
- **Primary Supervised Model**: {best_row['display_name']}
- **Character Error Rate (CER)**: {best_row['cer_pct']:.2f}%
- **Diacritic Error Rate (DER)**: {best_row['der_pct']:.2f}%
- **Improvement**: The fine-tuned model significantly reduced CER compared to the zero-shot baseline.

### Artifacts Generated
- **LoRA Adapter**: Uploaded to `Sam4rano/paddleocr-vl15-yoruba-lora`
- **Visuals**: Word Cloud, Error Rate Bar Charts, and Frequency Distributions saved in `results/tables/figures/`.
""")

## Step 11 — Inference with Fine-tuned Model

Use this code to test your uploaded model on a Yorùbá image crop.

In [ ]:
import torch
import os
import gc
from PIL import Image
from transformers import AutoProcessor, PaddleOCRVLForConditionalGeneration
from peft import PeftModel

# 0. Clear GPU memory to make room
gc.collect()
torch.cuda.empty_cache()

# 1. Configuration
base_model_id = "PaddlePaddle/PaddleOCR-VL-1.5"
adapter_model_id = "Sam4rano/paddleocr-vl15-yoruba-lora"

# Pick a sample image from your test set
test_images = os.listdir("data/processed/images/test")
test_image_path = os.path.join("data/processed/images/test", test_images[34])
print(f"Testing on: {test_image_path}")

# 2. Load Model and Processor
print("Loading base model and adapter (this may take a minute)...")
processor = AutoProcessor.from_pretrained(base_model_id, trust_remote_code=True)
model = PaddleOCRVLForConditionalGeneration.from_pretrained(
    base_model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Apply the LoRA adapter from Hugging Face
print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_model_id)
model.eval()

# 3. Run Inference
image = Image.open(test_image_path).convert("RGB")
prompt = "ocr the text in this image:"
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ],
    }
]

text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)
    # Trim the prompt from the output
    generated_ids = [ids[inputs.input_ids.shape[1]:] for ids in generated_ids]
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n--- OCR RESULT ---")
print(generated_text.strip())
display(image)